In [1]:
import requests
import uuid

In [6]:
def mcp_request(url: str, method: str, payload_params: dict = None, session_id: str = None, is_notification: bool = False, token: str = None):
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if session_id:
        headers["mcp-session-id"] = session_id
    if token:
        headers["Authorization"] = "Bearer " + token
        
    payload = {
        "jsonrpc": "2.0",
        "method": method,
    }
    if payload_params is not None:
        payload["params"] = payload_params
    if not is_notification:
        payload["id"] = "1"  # 只有请求带 id，通知不带

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=5)
        return response
    except Exception as e:
        return None

In [13]:
url = "http://127.0.0.1:8000/mcp"
token = "token_test"  # 替换为实际的 token
# 1) initialize（无会话头）
init_params = {
    "protocolVersion": "2025-03-26",
    "capabilities": {},
    "clientInfo": {"name": "mcp-gateway-test", "version": "0.1.0"}
}
init_resp = mcp_request(url, "initialize", payload_params=init_params, token=token)
session_id = init_resp.headers.get("mcp-session-id")

# 2) 发送 notifications/initialized（通知：不带 id，不需要 params）
mcp_request(url, "notifications/initialized", session_id=session_id, is_notification=True, token=token)

# 3) 后续请求（携带会话头）
resp = mcp_request(url, "tools/list", session_id=session_id, token=token)

# 4) 调用工具
params = {
    "name": "greet",
    "arguments": {
        "name": "test"
    }
}
resp = mcp_request(url, "tools/call", session_id=session_id, payload_params=params, token=token)

resp.json()

{'jsonrpc': '2.0',
 'id': '1',
 'result': {'content': [{'type': 'text', 'text': 'Hello, test!'}],
  'structuredContent': {'result': 'Hello, test!'},
  'isError': False}}

In [9]:
init_resp.text, resp.text

('Internal Server Error', 'Internal Server Error')

In [99]:
second_resp.text

'{"jsonrpc":"2.0","id":"1","error":{"code":-32602,"message":"Invalid request parameters","data":""}}'

In [82]:
response.url, response.headers

('http://127.0.0.1:8000/mcp',
 {'date': 'Sat, 06 Sep 2025 13:12:55 GMT', 'server': 'uvicorn', 'content-type': 'application/json', 'mcp-session-id': 'ee8910a289cc4d54832fa51ba1956be7', 'content-length': '99'})

In [16]:
import requests

def mcp_sse_request(url: str, method: str, payload_params: dict = None, session_id: str = None, token: str = None):
    headers = {
        "Content-Type": "application/json",
        "Accept": "text/event-stream",  # SSE 必须
    }
    if session_id:
        headers["mcp-session-id"] = session_id
    if token:
        headers["Authorization"] = "Bearer " + token

    payload = {
        "jsonrpc": "2.0",
        "method": method,
    }
    if payload_params is not None:
        payload["params"] = payload_params
    payload["id"] = "1"  # SSE 请求一般都带 id

    response = requests.get(url, headers=headers, json=payload, stream=True, timeout=10)
    # 逐行读取 SSE 响应
    for line in response.iter_lines():
        if line:
            print(line.decode())

# 服务端需用 mcp.run(transport="sse")
url = "http://127.0.0.1:8000/sse"  # SSE 路径通常为 /sse
token = "token_test"

# 1) initialize
init_params = {
    "protocolVersion": "2025-03-26",
    "capabilities": {},
    "clientInfo": {"name": "mcp-gateway-test", "version": "0.1.0"}
}
mcp_sse_request(url, "initialize", payload_params=init_params, token=token)

# 2) tools/list
mcp_sse_request(url, "tools/list", token=token)

# 3) tools/call
params = {
    "name": "greet",
    "arguments": {
        "name": "test"
    }
}
mcp_sse_request(url, "tools/call", payload_params=params, token=token)

event: endpoint
data: /messages/?session_id=141e7a6ce8da45f8b7a6bb4fd7c54ef2


KeyboardInterrupt: 

In [ ]:
from mcp.server.fastmcp import FastMCP
import mcp.types as types

mcp = FastMCP("StatefulServer", json_response=True)   # 默认 stateless_http=False

@mcp.tool()
def greet(name: str = "World") -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")   # 有状态，但客户端每次换新 ID 即可